# 2-1 원신호 전체 개관 (2단계 EDA 조각 ①)
시간영역에서 기본 정보를 먼저 파악한다. 해석과 결론은 [2-1_1113_원신호개관.md](2-1_1113_원신호개관.md)에 정리.

- 커널: **qe-phm** (VS Code 오른쪽 위에서 선택)
- 단위는 **파일**이다. 라벨이 파일마다 붙어 있으므로, 파일마다 통계를 하나씩 낸 뒤 정상 120개 vs 고장 30개를 비교한다
  (샘플 50,000개를 클래스별로 통째로 합쳐 describe하면 진폭 큰 파일 몇 개가 결과를 지배한다)
- 표시할 파일: 병목 고장 abn_002·022·029, 평균 -1 g 파일 nor_097·101·120·abn_016

In [ ]:
# 설정: 프로젝트 루트를 찾아 src를 불러온다 (어디서 실행해도 동작)
import os, sys
from pathlib import Path
_starts = [Path.cwd()] + ([Path(globals()["__vsc_ipynb_file__"]).parent] if "__vsc_ipynb_file__" in globals() else [])
ROOT_DIR = next(p for s in _starts for p in [s, *s.parents] if (p / "src" / "exp.py").exists())
sys.path.insert(0, str(ROOT_DIR))
from src.exp import *
from scipy.stats import kurtosis, skew
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

HERE = ROOT_DIR / "3_단계" / "2_EDA" / "참고" / "2-1_1113_원신호개관"
FIGS = HERE / "figs"; FIGS.mkdir(exist_ok=True)
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#222222", "#888888"   # 정상 파랑, 고장 주황
FS = 25000
MISSED = ["abn_002", "abn_022", "abn_029"]
MINUS1 = ["nor_097", "nor_101", "nor_120", "abn_016"]
PLAIN = FuncFormatter(lambda v, _: f"{v:g}")   # 로그축 눈금을 일반 숫자로 (마이너스 글꼴 깨짐 방지)
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

## 1. 데이터 확인
파일 수, 길이, 샘플링, 결측이 과제 설명과 맞는지 본다.

In [ ]:
X, y, names = load_train()
idx = {n: i for i, n in enumerate(names)}
print("배열 모양 (파일, 샘플):", X.shape, X.dtype)
print("정상", (y == 0).sum(), "개 / 고장", (y == 1).sum(), "개")
print("길이", X.shape[1] / FS, "초 (샘플링", FS, "Hz)")
print("결측(NaN)", np.isnan(X).sum(), "개 / 무한대", np.isinf(X).sum(), "개")
print("전체 최소·최대", X.min().round(3), X.max().round(3), "g")

## 2. 파일별 통계표
| 통계 | 뜻 |
|---|---|
| mean | 원신호 평균 (중력·센서 오프셋이 들어감) |
| std | 평균 제거 후 표준편차 = RMS. 전체 진동 세기 |
| min, max | 평균 제거 후 최소·최대 |
| max_abs | 평균 제거 후 최대 \|진폭\| (Peak) |
| p2p | 최대 - 최소 (peak-to-peak) |
| kurtosis | 첨도 (정규분포 = 0, 사인파 = -1.5). 뾰족한 충격이 있으면 커짐 |
| skew | 왜도. 한쪽으로 치우친 충격이면 0에서 멀어짐 |
| crest | 파고율 = max_abs / std. 바탕 대비 충격이 얼마나 튀나 |

In [ ]:
Xc = X - X.mean(1, keepdims=True)            # 평균 제거
S = pd.DataFrame({
    "name": names, "y": y,
    "mean": X.mean(1),
    "std": Xc.std(1),
    "min": Xc.min(1),
    "max": Xc.max(1),
    "max_abs": np.abs(Xc).max(1),
    "kurtosis": kurtosis(Xc, axis=1),
    "skew": skew(Xc, axis=1),
})
S["p2p"] = S["max"] - S["min"]
S["crest"] = S.max_abs / S["std"]
COLS = ["mean", "std", "min", "max", "max_abs", "p2p", "kurtosis", "skew", "crest"]
S.to_csv(FIGS / "2-1_stats.csv", index=False, encoding="utf-8-sig")
S.head()

## 3. 정상 vs 고장 describe
각 표의 한 행 = 통계 하나, 열 = 그 통계가 파일 120개(또는 30개)에 걸쳐 어떻게 분포하는가.

In [ ]:
print("정상 120개")
display(S[S.y == 0][COLS].describe().T)
print("고장 30개")
display(S[S.y == 1][COLS].describe().T)

### 한눈에: 중앙값 비교와 단변량 AUC
AUC = 고장 파일 하나와 정상 파일 하나를 뽑았을 때 고장 값이 더 클 확률 (0.5 = 구분 못 함, 1 또는 0 = 완전 구분).
설명용이며, 이 값으로 피처를 고르지 않는다.

In [ ]:
nor, ab = S[S.y == 0], S[S.y == 1]
summary = pd.DataFrame({
    "정상 5%": nor[COLS].quantile(.05), "정상 중앙값": nor[COLS].median(), "정상 95%": nor[COLS].quantile(.95),
    "고장 중앙값": ab[COLS].median(),
    "고장 중 정상95% 초과": [(ab[c] > nor[c].quantile(.95)).sum() for c in COLS],
    "AUC": [roc_auc_score(S.y, S[c]) for c in COLS],
})
summary

## 4. 이상치 파일 보기
평균·표준편차에서 튀는 파일이 누구인지 이름으로 확인한다.

In [ ]:
def extremes(col, k=5):
    s = S.sort_values(col)[["name", "y", col]]
    return pd.concat([s.head(k).assign(위치="가장 작음"), s.tail(k).assign(위치="가장 큼")])

for c in ["mean", "std", "max_abs"]:
    print(f"--- {c}")
    display(extremes(c))

## 5. 진폭이 크면 고장인가?
최대 |진폭|이 정상 95% 값을 넘는 파일이 정상·고장에 몇 개씩 있는지 센다.
정상도 5%(6개)는 정의상 넘으므로, "크면 고장"이 얼마나 맞는지 본다.

In [ ]:
thr = nor.max_abs.quantile(.95)
big = S.max_abs > thr
print(f"기준: 정상 95% = {thr:.3f} g")
display(pd.crosstab(S.y.map({0: "정상", 1: "고장"}), big.map({True: "기준 초과", False: "기준 이하"})))
print("기준을 넘는 정상:", ", ".join(S[big & (S.y == 0)].name))

## 6. 그림: 파일별 통계 분포
파란 띠 = 정상 5–95%, ○ = 병목 고장, □ = 평균 -1 g 파일. 표준편차·최대·첨도는 로그 축

In [ ]:
rng = np.random.default_rng(0)
jit = rng.uniform(-.18, .18, len(S))
fig, axs = plt.subplots(1, 4, figsize=(15, 4.6))
specs = [("mean", "평균 (g)", False), ("std", "표준편차 (g, 평균 제거)", True),
         ("max_abs", "최대 |진폭| (g, 평균 제거)", True), ("kurtosis", "첨도 (정규=0)", True)]
for a, (c, lab, logy) in zip(axs, specs):
    for g, col in ((0, BLUE), (1, ORANGE)):
        m = S.y == g
        a.scatter(g + jit[m], S.loc[m, c], s=14, color=col, alpha=.55, lw=0)
    a.axhspan(nor[c].quantile(.05), nor[c].quantile(.95), color=BLUE, alpha=.08, lw=0)
    for n in MISSED:
        i = idx[n]
        a.scatter(1 + jit[i], S.loc[i, c], s=60, facecolor="none", edgecolor=INK, lw=1.2)
        a.annotate(n[4:], (1 + jit[i], S.loc[i, c]), xytext=(6, 0), textcoords="offset points",
                   fontsize=7, color=INK, va="center")
    for n in MINUS1:
        i = idx[n]
        a.scatter(S.y[i] + jit[i], S.loc[i, c], s=60, marker="s", facecolor="none", edgecolor=MUTED, lw=1)
    if c == "kurtosis":
        a.set_yscale("symlog", linthresh=1); a.set_ylim(-1.5, None)
    elif logy:
        a.set_yscale("log")
    if logy:
        a.yaxis.set_major_formatter(PLAIN)
    a.set_xticks([0, 1]); a.set_xticklabels([f"정상 {(S.y == 0).sum()}", f"고장 {(S.y == 1).sum()}"])
    a.set_xlim(-.6, 1.6); a.set_title(lab, fontsize=10, loc="left")
    a.grid(axis="y", alpha=.25); a.spines[["top", "right"]].set_visible(False)
fig.suptitle("파일별 원신호 통계 — 파란 띠 = 정상 5–95%, ○ = 병목 고장 002·022·029, □ = 평균 -1 g 파일",
             fontsize=11, x=.01, ha="left")
fig.tight_layout()
fig.savefig(FIGS / "2-1_stats.png", dpi=130)
plt.show()

### 표준편차 × 첨도
두 통계를 함께 보면 갈리는가 (파일 하나 = 점)

In [ ]:
fig, a = plt.subplots(figsize=(7.5, 5.5))
for g, col, lab in ((0, BLUE, "정상"), (1, ORANGE, "고장")):
    m = S.y == g
    a.scatter(S.loc[m, "std"], S.loc[m, "kurtosis"], s=18, color=col, alpha=.65, lw=0, label=lab)
for n in MISSED + MINUS1 + ["nor_001", "nor_050", "abn_015", "abn_017"]:
    i = idx[n]
    a.annotate(n, (S.loc[i, "std"], S.loc[i, "kurtosis"]), xytext=(5, 3), textcoords="offset points",
               fontsize=7, color=INK)
a.set_xscale("log"); a.set_yscale("symlog", linthresh=1); a.set_ylim(-1.5, None)
a.xaxis.set_major_formatter(PLAIN); a.yaxis.set_major_formatter(PLAIN)
a.set_xlabel("표준편차 (g)"); a.set_ylabel("첨도")
a.legend(frameon=False); a.grid(alpha=.25); a.spines[["top", "right"]].set_visible(False)
a.set_title("표준편차 × 첨도 (파일 하나 = 점)", loc="left", fontsize=11)
fig.tight_layout()
fig.savefig(FIGS / "2-1_std_kurt.png", dpi=130)
plt.show()

## 7. 150개 전체 파형 한눈에
파일마다 y축을 따로 잡아 **모양**을 본다 (크기 비교는 위 통계 그림으로).
50,000점을 25점씩 묶어 묶음마다 최소~최대를 칠한다 → 2,000칸으로 줄여도 순간 충격이 사라지지 않는다.
제목 = 파일 이름 · 최대 |진폭| (g) · 첨도. ○ 병목 고장과 □ -1 g 파일은 제목을 굵게.

In [ ]:
t_env = (np.arange(X.shape[1] // 25) * 25 + 12.5) / FS   # 묶음 가운데 시각

def envelope(x, k=25):
    x = x[: len(x) // k * k].reshape(-1, k)
    return x.min(1), x.max(1)

def wave_grid(sel, ncol, color, title, fname):
    nrow = int(np.ceil(len(sel) / ncol))
    fig, axs = plt.subplots(nrow, ncol, figsize=(2.3 * ncol, 1.15 * nrow), sharex=True)
    for a in axs.flat:
        a.set_axis_off()
    for a, i in zip(axs.flat, sel):
        lo, hi = envelope(Xc[i])
        a.fill_between(t_env, lo, hi, color=color, lw=0)
        a.set_axis_on(); a.set_yticks([]); a.tick_params(labelsize=5, length=2)
        a.spines[["top", "right", "left"]].set_visible(False)
        n = names[i]
        a.set_title(f"{n}  {S['max_abs'][i]:.2f}g  k{S['kurtosis'][i]:.0f}", fontsize=6.5, loc="left", pad=2,
                    fontweight="bold" if n in MISSED + MINUS1 else "normal")
    fig.suptitle(title, fontsize=11, x=.01, ha="left")
    fig.tight_layout(h_pad=.4, w_pad=.3)
    fig.savefig(FIGS / fname, dpi=110)
    plt.show()

wave_grid(np.where(y == 0)[0], 10, BLUE, "정상 120개 파형 (평균 제거, y축 파일별, 가로 0–2초)", "2-1_wave_all_normal.png")
wave_grid(np.where(y == 1)[0], 6, ORANGE, "고장 30개 파형 (평균 제거, y축 파일별, 가로 0–2초)", "2-1_wave_all_abnormal.png")

## 8. 대표 파일 파형 (2초 전체 · 0.1초 확대)
대표 = 표준편차가 중앙값에 가장 가까운 정상·고장. 그 밖에 큰 충격 정상(nor_001·050), -1 g 파일, 병목 고장.
보고 싶은 파일이 있으면 `SHOW`에 이름을 넣고 다시 실행한다.

In [ ]:
def nearest_median(g):
    s = S[S.y == g]
    return s.iloc[(s["std"] - s["std"].median()).abs().argsort().iloc[0]]["name"]

rep_nor, rep_ab = nearest_median(0), nearest_median(1)
print("대표 정상", rep_nor, "/ 대표 고장", rep_ab)
SHOW = [rep_nor, "nor_001", "nor_050", "nor_097", rep_ab, "abn_016"] + MISSED
t = np.arange(X.shape[1]) / FS

def wave_fig(show, t0, t1, title, fname=None):
    m = (t >= t0) & (t <= t1)
    fig, axs = plt.subplots(len(show), 1, figsize=(13, 1.35 * len(show)), sharex=True, squeeze=False)
    for a, n in zip(axs[:, 0], show):
        i = idx[n]
        a.plot(t[m], Xc[i, m], lw=.4, color=ORANGE if y[i] else BLUE)
        r = S.iloc[i]
        a.set_title(f"{n}   평균 {r['mean']:+.2f} g · sd {r['std']:.3f} · 최대 {r['max_abs']:.2f} · 첨도 {r['kurtosis']:.1f}",
                    fontsize=8, loc="left", color=INK)
        a.grid(alpha=.2); a.spines[["top", "right"]].set_visible(False); a.tick_params(labelsize=7)
    axs[-1, 0].set_xlabel("초")
    fig.suptitle(title, fontsize=11, x=.01, ha="left")
    fig.tight_layout()
    if fname:
        fig.savefig(FIGS / fname, dpi=120)
    plt.show()

wave_fig(SHOW, 0, 2, "파형 2초 전체 (평균 제거, y축 파일별) — 파랑 정상 · 주황 고장", "2-1_wave_full.png")

In [ ]:
wave_fig(SHOW, 1.0, 1.1, "파형 확대 1.0–1.1초 (평균 제거, y축 파일별)", "2-1_wave_zoom.png")

### 같은 y축으로 비교
대표 정상 · 대표 고장 · 병목 고장의 진폭 크기를 직접 비교

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(13, 5.5), sharex=True, sharey=True)
for a, n in zip(axs, [rep_nor, rep_ab, "abn_022"]):
    i = idx[n]
    a.plot(t, Xc[i], lw=.3, color=ORANGE if y[i] else BLUE)
    a.set_title(n, fontsize=8, loc="left"); a.grid(alpha=.2); a.spines[["top", "right"]].set_visible(False)
axs[-1].set_xlabel("초")
fig.suptitle("같은 y축으로 본 대표 정상 · 대표 고장 · 병목 고장", fontsize=11, x=.01, ha="left")
fig.tight_layout()
fig.savefig(FIGS / "2-1_wave_sharey.png", dpi=120)
plt.show()

## 9. 파일별 파형 150장 → `3_단계/2_EDA/참고/2-1_1113_원신호개관/파형_150/`
파일 하나 = 그림 하나 (`nor_001.png` …). 축소판이 아니라 50,000점 전부를 그린다.
- 위: 2초 전체. 점선 = 정상 최대 |진폭|의 95% 값(±1.09 g)이 y축 범위 안에 들어올 때만 표시
- 아래: 최대 |진폭| 지점 앞뒤 40 ms 확대 (충격 하나의 모양)
- 제목: 라벨, 평균, 표준편차·최대 |진폭|·첨도·파고율과 각각의 정상 대비 백분위
1–2분 걸린다. 다시 만들 필요가 없으면 건너뛴다.

In [ ]:
WAVE_DIR = HERE / "파형_150"; WAVE_DIR.mkdir(exist_ok=True)
thr95 = nor.max_abs.quantile(.95)
pct = lambda c, v: (nor[c] < v).mean() * 100

def one_file(i):
    n, lab = names[i], ("고장" if y[i] else "정상")
    col = ORANGE if y[i] else BLUE
    r = S.iloc[i]
    fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 5.6), gridspec_kw={"height_ratios": [1.6, 1]})
    a1.plot(t, Xc[i], lw=.35, color=col)
    lim = np.abs(Xc[i]).max() * 1.08
    a1.set_ylim(-lim, lim)
    if thr95 < lim:
        for s in (1, -1):
            a1.axhline(s * thr95, color=MUTED, lw=.8, ls="--")
        a1.text(2.0, thr95, f" 정상 95% {thr95:.2f} g", fontsize=7, color=MUTED, va="bottom", ha="right")
    k = int(np.abs(Xc[i]).argmax()); w = int(0.02 * FS)
    lo, hi = max(0, k - w), min(X.shape[1], k + w)
    a1.axvspan(t[lo], t[hi - 1], color=MUTED, alpha=.15, lw=0)
    a1.set_xlim(0, 2); a1.set_xlabel("초", fontsize=8); a1.set_ylabel("g (평균 제거)", fontsize=8)
    a2.plot(t[lo:hi] * 1000, Xc[i, lo:hi], lw=.7, color=col)
    a2.set_xlabel("ms", fontsize=8); a2.set_ylabel("g", fontsize=8)
    a2.set_title(f"최대 |진폭| 지점 {t[k]:.3f}초 앞뒤 20 ms 확대 (위 그림의 회색 구간)", fontsize=8, loc="left")
    for a in (a1, a2):
        a.grid(alpha=.2); a.spines[["top", "right"]].set_visible(False); a.tick_params(labelsize=7)
    a1.set_title(f"{n} ({lab})   평균 {r['mean']:+.3f} g   |   "
                 f"sd {r['std']:.3f} ({pct('std', r['std']):.0f}%) · 최대 {r['max_abs']:.2f} ({pct('max_abs', r['max_abs']):.0f}%) · "
                 f"첨도 {r['kurtosis']:.1f} ({pct('kurtosis', r['kurtosis']):.0f}%) · 파고율 {r['crest']:.1f} ({pct('crest', r['crest']):.0f}%)"
                 "   ( ) = 정상 120개 중 백분위", fontsize=9, loc="left")
    fig.tight_layout()
    fig.savefig(WAVE_DIR / f"{n}.png", dpi=100)
    plt.close(fig)

for i in range(len(names)):
    one_file(i)
print(len(list(WAVE_DIR.glob("*.png"))), "장 저장:", WAVE_DIR)

## 자유 탐색
궁금한 파일·구간을 여기서 바로 본다. 예: `wave_fig(['abn_022', 'nor_006'], 0.9, 1.1, '확대')`

In [ ]:
wave_fig(["abn_022", "nor_006"], 0.9, 1.1, "자유 탐색 예시")